# Preprocessing & Entraînement — V1 (Baseline)

Ce notebook constitue la **baseline** du projet avec 7 features simples. Le notebook `03_feature_engineering.ipynb` pousse le R² de 0.50 à 0.68 avec un feature set enrichi (17 features).

In [4]:
import pathlib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

MODELS_DIR = pathlib.Path("../models")
DATA_PATH  = pathlib.Path("../sncf_retards.csv")
TARGET = "Retard moyen de tous les trains à l'arrivée"


## 1. Chargement et nettoyage

In [5]:
df = pd.read_csv(DATA_PATH, sep=";")

cols_drop = [c for c in df.columns if "Commentaire" in c] + ["Service"]
df = df.drop(columns=cols_drop)

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["Année"] = df["Date"].dt.year
df["Mois"]  = df["Date"].dt.month
df = df.drop(columns=["Date"])

print(f"Avant nettoyage : {len(df)} lignes")
df = df[df[TARGET] >= 0]
print(f"Après nettoyage  : {len(df)} lignes (retards négatifs supprimés)")
df.head()


Avant nettoyage : 12181 lignes
Après nettoyage  : 12061 lignes (retards négatifs supprimés)


,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",Année,Mois
0,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,289,11.247809,3.693179,147,28.436735,...,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336,2018,1
1,LE MANS,PARIS MONTPARNASSE,56,406,1,213,8.479969,4.567119,105,18.049048,...,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333,2018,1
2,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,21,6.239683,0.286283,19,24.736842,...,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111,2018,1
3,PARIS MONTPARNASSE,NANTES,124,508,3,71,7.235211,0.734290,58,33.726437,...,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852,2018,1
4,POITIERS,PARIS MONTPARNASSE,94,472,4,224,6.784673,3.229701,89,14.592697,...,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386,2018,1


## 2. Features et split train/test

In [6]:
FEATURES_V1 = [
    "Gare_depart_enc",
    "Gare_arrivee_enc",
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
    "Nombre de trains annulés",
    "Année",
    "Mois",
]

X_raw = df.copy()
y = df[TARGET]

# Split avant tout encodage pour éviter le data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Encodage des gares — fitté uniquement sur le train set
le_depart  = LabelEncoder().fit(X_train_raw["Gare de départ"])
le_arrivee = LabelEncoder().fit(X_train_raw["Gare d'arrivée"])

for split in [X_train_raw, X_test_raw]:
    split["Gare_depart_enc"]  = le_depart.transform(split["Gare de départ"])
    split["Gare_arrivee_enc"] = le_arrivee.transform(split["Gare d'arrivée"])

X_train = X_train_raw[FEATURES_V1]
X_test  = X_test_raw[FEATURES_V1]

joblib.dump(le_depart,  MODELS_DIR / "le_depart.joblib")
joblib.dump(le_arrivee, MODELS_DIR / "le_arrivee.joblib")

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")


Train : 9648 lignes | Test : 2413 lignes


## 3. Entraînement des modèles

In [7]:
models_v1 = {
    "linear_regression": LinearRegression(),
    "random_forest":     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    "adaboost":          AdaBoostRegressor(n_estimators=200, random_state=42),
    "svr":               Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", C=10))]),
    "knn":               Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=10))]),
}

for name, model in models_v1.items():
    print(f"Entraînement : {name}...")
    model.fit(X_train, y_train)
    joblib.dump(model, MODELS_DIR / f"{name}.joblib")

print("\nTous les modèles entraînés et sauvegardés.")


Entraînement : linear_regression...
Entraînement : random_forest...
Entraînement : gradient_boosting...
Entraînement : adaboost...
Entraînement : svr...
Entraînement : knn...

Tous les modèles entraînés et sauvegardés.


## 4. Évaluation

In [8]:
print(f"{'Modèle':<25} {'MAE':>8} {'RMSE':>8} {'R² train':>10} {'R² test':>10}")
print("-" * 70)

results = []
for name, model in models_v1.items():
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_train = r2_score(y_train, y_pred_train)
    r2_test  = r2_score(y_test,  y_pred_test)

    print(f"{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2_train:>10.4f} {r2_test:>10.4f}")
    results.append({"Modèle": name, "MAE": mae, "RMSE": rmse, "R2_train": r2_train, "R2_test": r2_test})

results_df = pd.DataFrame(results).sort_values("R2_test", ascending=False)
results_df.to_csv(MODELS_DIR / "metrics_v1.csv", index=False)
print("\nMétriques sauvegardées dans models/metrics_v1.csv")


Modèle                         MAE     RMSE   R² train    R² test
----------------------------------------------------------------------
linear_regression             2.66     3.59     0.1528     0.1482
random_forest                 1.89     2.77     0.9216     0.4928
gradient_boosting             1.89     2.74     0.7485     0.5038
adaboost                      3.12     3.96     0.1142    -0.0330
svr                           2.18     3.17     0.3288     0.3389
knn                           2.17     3.06     0.4674     0.3805

Métriques sauvegardées dans models/metrics_v1.csv
